In [1]:
!pip install ipynb

In [2]:
%run data_collect.ipynb
%run pre_process.ipynb

import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.6/57.6 MB 593.1 kB/s eta 0:00:00m eta 0:00:010:00:03
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.0/216.0 MB 552.8 kB/s eta 0:00:00m eta 0:00:010:00:10


In [3]:
collector = DataCollector()

expected_rows = 5110
expected_cols = [
    "id",
    "gender",
    "age",
    "hypertension",
    "heart_disease",
    "ever_married",
    "work_type",
    "Residence_type",
    "avg_glucose_level",
    "bmi",
    "smoking_status",
    "stroke",
]

X_splits, y_splits = collector.split_data(expected_rows, expected_cols)

Partition row count match with original dataset.
Class proportions reasonably preserved.


In [4]:
class Linear_Base_Model:

    def __init__(
        self,
        X_train_processed,
        y_train_processed,
        X_val_processed,
        y_val_processed,
    ):
        self.X_train_processed = X_train_processed
        self.y_train_processed = y_train_processed
        self.processed_train = pd.concat(
            [self.X_train_processed, self.y_train_processed], axis=1
        )

        self.X_val = X_val_processed
        self.y_val = y_val_processed

        # Fit all linear / distance-based models
        self._fit_logistic_regression()
        self._fit_svm()
        self._fit_knn()

        # Measure performance for all models
        self._performance_measure()

    def _fit_logistic_regression(self):
        self.lr_model = LogisticRegression(
            class_weight="balanced", random_state=42, max_iter=1000
        )
        self.lr_model.fit(self.X_train_processed, self.y_train_processed)

    def _fit_svm(self):
        self.svm_model = SVC(
            class_weight="balanced", probability=True, random_state=42
        )
        self.svm_model.fit(self.X_train_processed, self.y_train_processed)

    def _fit_knn(self):
        self.knn_model = KNeighborsClassifier(n_neighbors=5)
        self.knn_model.fit(self.X_train_processed, self.y_train_processed)

    def _performance_measure(self):
        models = {
            "Logistic Regression": self.lr_model,
            "Support Vector Machine": self.svm_model,
            "K-Nearest Neighbors": self.knn_model,
        }

        for name, model in models.items():
            preds = model.predict(self.X_val)
            probs = model.predict_proba(self.X_val)[:, 1]

            print(f"----------{name} Base Model--------------")
            print(f"Accuracy:  {accuracy_score(self.y_val, preds):.4f}")
            print(
                f"Precision: {precision_score(self.y_val, preds, zero_division=0):.4f}"
            )
            print(
                f"Recall:    {recall_score(self.y_val, preds, zero_division=0):.4f}"
            )
            print(
                f"F1-Score:  {f1_score(self.y_val, preds, zero_division=0):.4f}"
            )
            print(f"ROC-AUC:   {roc_auc_score(self.y_val, probs):.4f}\n")

In [5]:
class Tree_Base_Model:

    def __init__(
        self,
        X_train_processed,
        y_train_processed,
        X_val_processed,
        y_val_processed,
    ):
        self.X_train_processed = X_train_processed
        self.y_train_processed = y_train_processed
        self.processed_train = pd.concat(
            [self.X_train_processed, self.y_train_processed], axis=1
        )

        self.X_val = X_val_processed
        self.y_val = y_val_processed

        # Calculate positive scale factor for XGBoost class balancing
        self.scale_pos_weight = (self.y_train_processed == 0).sum() / (
            self.y_train_processed == 1
        ).sum()

        # Fit all tree-based models
        self._fit_decision_tree()
        self._fit_random_forest()
        self._fit_xgboost()

        # Measure performance for all models
        self._performance_measure()

    def _fit_decision_tree(self):
        self.dt_model = DecisionTreeClassifier(
            class_weight="balanced", random_state=42
        )
        self.dt_model.fit(self.X_train_processed, self.y_train_processed)

    def _fit_random_forest(self):
        self.rf_model = RandomForestClassifier(
            class_weight="balanced", random_state=42
        )
        self.rf_model.fit(self.X_train_processed, self.y_train_processed)

    def _fit_xgboost(self):
        self.xgb_model = XGBClassifier(
            scale_pos_weight=self.scale_pos_weight,
            random_state=42,
            eval_metric="logloss",
        )
        self.xgb_model.fit(self.X_train_processed, self.y_train_processed)

    def _performance_measure(self):
        models = {
            "Decision Tree": self.dt_model,
            "Random Forest": self.rf_model,
            "XGBoost": self.xgb_model,
        }

        for name, model in models.items():
            preds = model.predict(self.X_val)
            probs = model.predict_proba(self.X_val)[:, 1]

            print(f"----------{name} Base Model--------------")
            print(f"Accuracy:  {accuracy_score(self.y_val, preds):.4f}")
            print(
                f"Precision: {precision_score(self.y_val, preds, zero_division=0):.4f}"
            )
            print(
                f"Recall:    {recall_score(self.y_val, preds, zero_division=0):.4f}"
            )
            print(
                f"F1-Score:  {f1_score(self.y_val, preds, zero_division=0):.4f}"
            )
            print(f"ROC-AUC:   {roc_auc_score(self.y_val, probs):.4f}\n")

In [6]:
lr_processor = Pre_Processor(model_type="linear")

X_train_lr, y_train_lr = lr_processor.fit_transform(
    X_splits["X_train"], y_splits["y_train"]
)
X_val_lr, y_val_lr = lr_processor.transform(
    X_splits["X_val"], y_splits["y_val"]
)

# Run Logistic Regression
linear_base = Linear_Base_Model(
    X_train_lr, y_train_lr, X_val_lr, y_val_lr
)


# 5. Preprocess data for Decision Tree
dt_processor = Pre_Processor(model_type="tree")

X_train_dt, y_train_dt = dt_processor.fit_transform(
    X_splits["X_train"], y_splits["y_train"]
)
X_val_dt, y_val_dt = dt_processor.transform(
    X_splits["X_val"], y_splits["y_val"]
)

# Run Decision Tree
tree_base = Tree_Base_Model(X_train_dt, y_train_dt, X_val_dt, y_val_dt)

/home/alvaro/AAI501-Alvaro-Folgueira-Project/.venv/lib/python3.12/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


----------Logistic Regression Base Model--------------
Accuracy:  0.7154
Precision: 0.1213
Recall:    0.7838
F1-Score:  0.2101
ROC-AUC:   0.8418

----------Support Vector Machine Base Model--------------
Accuracy:  0.7441
Precision: 0.1122
Recall:    0.6216
F1-Score:  0.1901
ROC-AUC:   0.7718

----------K-Nearest Neighbors Base Model--------------
Accuracy:  0.9465
Precision: 0.0000
Recall:    0.0000
F1-Score:  0.0000
ROC-AUC:   0.5643

----------Decision Tree Base Model--------------
Accuracy:  0.9282
Precision: 0.0909
Recall:    0.0541
F1-Score:  0.0678
ROC-AUC:   0.5133

----------Random Forest Base Model--------------
Accuracy:  0.9347
Precision: 0.1905
Recall:    0.1081
F1-Score:  0.1379
ROC-AUC:   0.7908

----------XGBoost Base Model--------------
Accuracy:  0.9334
Precision: 0.2667
Recall:    0.2162
F1-Score:  0.2388
ROC-AUC:   0.7893

